# Required library

In [1]:
from google.colab import files

In [2]:
import os

In [3]:
import io

In [4]:
import zipfile

In [5]:
import tempfile

In [6]:
import pandas as pd

# Upload the zip file

In [7]:
# Create an object to store uploaded zip files
uploaded = files.upload()
# Extract the zip file content
zip_filename = next(iter(uploaded.keys()))
# Provide the user some feedback
print(f"Uploaded: {zip_filename}")

Saving combined_results_20260914_012507.zip to combined_results_20260914_012507.zip
Uploaded: combined_results_20260914_012507.zip


# Extract the ZIP file content

In [8]:
# Create a temporary directory

temp_dir = tempfile.TemporaryDirectory()

extract_path = temp_dir.name

# Extract the ZIP

with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("ZIP extracted to:")
print(extract_path)

print("\nFiles found:")

for file in os.listdir(extract_path):
    print(file)

ZIP extracted to:
/tmp/tmp2remfgnz

Files found:
FINAL_failure.csv
FINAL_success.csv
FINAL_partial.csv


# Load the CSVs

In [9]:
if not uploaded:
    raise ValueError("No file was uploaded.")

# Take the first (and expected only) uploaded file
zip_filename = next(iter(uploaded))
zip_bytes = uploaded[zip_filename]

if not zip_filename.lower().endswith(".zip"):
    raise ValueError(f"Expected a .zip file, got: {zip_filename}")

dataframes = {}
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
    csv_names = [n for n in zf.namelist() if n.lower().endswith(".csv")]

    if not csv_names:
        raise ValueError("No CSV files found inside the zip.")

    for name in csv_names:
        with zf.open(name) as f:
            df = pd.read_csv(f)
            # Use just the base filename as the key (drop any folder prefix)
            key = name.split("/")[-1]
            dataframes[key] = df
            print(f"Loaded '{key}' -> shape {df.shape}")

Loaded 'FINAL_success.csv' -> shape (203, 13)
Loaded 'FINAL_partial.csv' -> shape (0, 13)
Loaded 'FINAL_failure.csv' -> shape (33, 13)


In [10]:
dataframes.keys()

dict_keys(['FINAL_success.csv', 'FINAL_partial.csv', 'FINAL_failure.csv'])

In [11]:
# Load datasets for verficaiton
success_df = dataframes['FINAL_success.csv']
failure_df = dataframes['FINAL_failure.csv']
partial_df = dataframes['FINAL_partial.csv']

In [12]:
# Verify output
success_df

,address,normalized_address,city,province,country,status,postal_code,latitude,longitude,display_name,error_type,error_message,normalization_changes
0,1399 BERKLEY DR NW,1399 BERKLEY DRIVE NW,Calgary,Alberta,Canada,SUCCESS,T2K 1T5,51.128814,-114.100048,"1399, Berkley Drive NW, Beddington Heights, Ca...",NaN,NaN,Expanded street type: DR -> DRIVE
1,732 ARBOUR LAKE DR NW,732 ARBOUR LAKE DRIVE NW,Calgary,Alberta,Canada,SUCCESS,T3G 3Y8,51.135133,-114.192882,"732, Arbour Lake Drive NW, Arbour Lake, Calgar...",NaN,NaN,Expanded street type: DR -> DRIVE
2,51 FALWOOD WY NE,51 FALWOOD WAY NE,Calgary,Alberta,Canada,SUCCESS,T1Y 5G7,51.098386,-113.943812,"51, Falwood Way NE, Falconridge, Calgary, Albe...",NaN,NaN,Expanded street type: WY -> WAY
3,436 RANCHRIDGE CO NW,436 RANCHRIDGE COURT NW,Calgary,Alberta,Canada,SUCCESS,T3G 2B3,51.121319,-114.183314,"600 Ranchridge Court NW, Ranchlands, Calgary, ...",NaN,NaN,Expanded street type: CO -> COURT
4,68 SETON GV SE,68 SETON GROVE SE,Calgary,Alberta,Canada,SUCCESS,T3M 3P6,50.866710,-113.957799,"68, Seton Grove SE, Seton, Calgary, Alberta, T...",NaN,NaN,Expanded street type: GV -> GROVE
...,...,...,...,...,...,...,...,...,...,...,...,...,...
198,35 HANOVER RD SW,35 HANOVER ROAD SW,Calgary,Alberta,Canada,SUCCESS,T2V 0Y1,50.971997,-114.089670,"35, Hanover Road SW, Haysboro, Calgary, Albert...",NaN,NaN,Expanded street type: RD -> ROAD
199,127 SUNMILLS DR SE,127 SUNMILLS DRIVE SE,Calgary,Alberta,Canada,SUCCESS,T2X 2R1,50.902377,-114.048568,"127, Sunmills Drive SE, Sundance, Calgary, Alb...",NaN,NaN,Expanded street type: DR -> DRIVE
200,9880 2 AV SW,9880 2 AVENUE SW,Calgary,Alberta,Canada,SUCCESS,T3C 0A2,51.051187,-114.139323,"2 Avenue SW, Spruce Cliff, Calgary, Alberta, T...",NaN,NaN,Expanded street type: AV -> AVENUE
201,569 RIVERBEND DR SE,569 RIVERBEND DRIVE SE,Calgary,Alberta,Canada,SUCCESS,T2C 3S7,50.979622,-114.025362,"569, Riverbend Drive SE, Riverbend, Calgary, A...",NaN,NaN,Expanded street type: DR -> DRIVE


In [13]:
# Verify output
failure_df

,address,normalized_address,city,province,country,status,postal_code,latitude,longitude,display_name,error_type,error_message,normalization_changes
0,5925 117 ST NW,5925 117 STREET NW,Calgary,Alberta,Canada,FAILURE,NaN,NaN,NaN,NaN,NoMatch,Address not found.,Expanded street type: ST -> STREET
1,35 EVERRIDGE VI SW,35 EVERRIDGE VIEW SW,Calgary,Alberta,Canada,FAILURE,NaN,NaN,NaN,NaN,NoMatch,Address not found.,Expanded street type: VI -> VIEW
2,18 VALLEY WOODS LD NW,18 VALLEY WOODS LD NW,Calgary,Alberta,Canada,FAILURE,NaN,NaN,NaN,NaN,NoMatch,Address not found.,NaN
3,30 BRIGHTONCREST CV SE,30 BRIGHTONCREST CV SE,Calgary,Alberta,Canada,FAILURE,NaN,NaN,NaN,NaN,NoMatch,Address not found.,NaN
4,#1406 28 RED EMBERS PZ NE,#1406 28 RED EMBERS PZ NE,Calgary,Alberta,Canada,FAILURE,NaN,NaN,NaN,NaN,NoMatch,Address not found.,NaN
5,65 COVILLE BA NE,65 COVILLE BA NE,Calgary,Alberta,Canada,FAILURE,NaN,NaN,NaN,NaN,NoMatch,Address not found.,NaN
6,214 PANORAMA HILLS TC NW,214 PANORAMA HILLS TC NW,Calgary,Alberta,Canada,FAILURE,NaN,NaN,NaN,NaN,NoMatch,Address not found.,NaN
7,13 SHAWNEE HE SW,13 SHAWNEE HEIGHTS SW,Calgary,Alberta,Canada,FAILURE,NaN,NaN,NaN,NaN,NoMatch,Address not found.,Expanded street type: HE -> HEIGHTS
8,#101 2117 81 ST SW,#101 2117 81 STREET SW,Calgary,Alberta,Canada,FAILURE,NaN,NaN,NaN,NaN,NoMatch,Address not found.,Expanded street type: ST -> STREET
9,56 MIDPARK GD SE,56 MIDPARK GD SE,Calgary,Alberta,Canada,FAILURE,NaN,NaN,NaN,NaN,NoMatch,Address not found.,NaN


In [14]:
# Verify output
partial_df

,address,normalized_address,city,province,country,status,postal_code,latitude,longitude,display_name,error_type,error_message,normalization_changes
